In [1]:
##initialization and imports
%load_ext autoreload
%autoreload 2

import sys
import os

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import ee
import geemap
import pandas as pd

from src.config import initialize_earth_engine, PRE_FIRE_START, PRE_FIRE_END, POST_FIRE_START, POST_FIRE_END
from src.data_loaders import get_susa_valley_aoi, get_bounding_box_extent, get_sentinel2_collection
from src.processors import calculate_ndvi, calculate_dnbr
from src.map_builder import create_base_map, add_sentinel2_layer, add_index_layer
from src.visualizers import plot_dnbr_histogram, plot_ndvi_scatterplot

initialize_earth_engine()

# 1. Load BOTH boundaries: CEMS irregular polygon (for science) & Bounding Box (for map framing)
cems_boundary = get_susa_valley_aoi()
aoi_bbox = get_bounding_box_extent()

print("[INFO] Setup complete with CEMS boundary and bounding box!")

C:\Users\MARYAM\.conda\envs\smartsat_env\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.15) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


[INFO] Google Earth Engine initialized successfully.
[INFO] Successfully loaded official CEMS irregular boundary from: E:\GeoSpatialProjects\MastersThesisProject\SusaValleyWFAnalysis\resources\EMSR253_susa_del_2017_converted.geojson
[INFO] Setup complete with CEMS boundary and bounding box!


In [2]:
##Data Loading & Processing and clipping (Indices)
# Load image collections using the broader bounding box to capture context
pre_col = get_sentinel2_collection(aoi_bbox, PRE_FIRE_START, PRE_FIRE_END)
post_col = get_sentinel2_collection(aoi_bbox, POST_FIRE_START, POST_FIRE_END)

pre_fire_image = pre_col.median()
post_fire_image = post_col.median()

# Compute indices
pre_ndvi = calculate_ndvi(pre_fire_image).select('NDVI')
post_ndvi = calculate_ndvi(post_fire_image).select('NDVI')
dnbr_image = calculate_dnbr(pre_fire_image, post_fire_image)

print("[INFO] Rasters computed successfully!")

[INFO] Rasters computed successfully!


In [3]:
# Interactive Map Visualization
Map = create_base_map()
Map.centerObject(aoi_bbox, 11)  # Center nicely using the broader box

# Add raster layers clipped/visualized
add_sentinel2_layer(Map, post_fire_image, name="Post-Fire True Color (S2)")

ndvi_palette = ['brown', 'yellow', 'lightgreen', 'green', 'darkgreen']
add_index_layer(Map, pre_ndvi, 'NDVI', ndvi_palette, -0.2, 0.8, 'Pre-Fire NDVI')
add_index_layer(Map, post_ndvi, 'NDVI', ndvi_palette, -0.2, 0.8, 'Post-Fire NDVI')

burn_palette = ['blue', 'gray', 'yellow', 'orange', 'red', 'purple']
add_index_layer(Map, dnbr_image, 'dNBR', burn_palette, -0.1, 0.6, 'Burn Severity (dNBR)')

# --- ADD VECTOR OVERLAYS ---
# 1. True CEMS Irregular Burn Scar (Red Outline)
Map.addLayer(cems_boundary, {'color': 'red', 'fillColor': '00000000', 'width': 2}, 'CEMS Official Burn Perimeter')

# 2. Study Bounding Box (Subtle Gray Frame)
Map.addLayer(aoi_bbox, {'color': 'gray', 'fillColor': '00000000', 'width': 1}, 'Study Bounding Box')

Map.addLayerControl()
Map

Map(center=[45.167219197090695, 7.119143625060395], controls=(WidgetControl(options=['position', 'transparent_…

In [4]:
# Optional: Clip the split view layers to the exact CEMS boundary if desired
pre_ndvi_clipped = pre_ndvi.clip(cems_boundary)
post_ndvi_clipped = post_ndvi.clip(cems_boundary)

left_layer = geemap.ee_tile_layer(
    pre_ndvi.clip(cems_boundary),
    {'min': -0.2, 'max': 0.8, 'palette': ndvi_palette},
    'Pre-Fire NDVI'
)
right_layer = geemap.ee_tile_layer(post_ndvi_clipped, {'min': -0.2, 'max': 0.8, 'palette': ndvi_palette}, 'Post-Fire NDVI')

split_map = geemap.Map(center=[45.167, 7.119], zoom=12)
split_map.split_map(left_layer=left_layer, right_layer=right_layer)

# Add CEMS boundary to split map
split_map.addLayer(cems_boundary, {'color': 'red', 'fillColor': '00000000', 'width': 2}, 'CEMS Burn Perimeter')
split_map

Map(center=[45.167, 7.119], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

In [ ]:
# 1. Sample pixels strictly inside the official CEMS polygon
analysis_image = ee.Image.cat([
    pre_ndvi.rename('pre_ndvi'),
    post_ndvi.rename('post_ndvi'),
    dnbr_image.select(['dNBR']).rename('dnbr')
]).clip(cems_boundary)

# Optimize sampling for speed and performance
sampled_fc = analysis_image.sample(
    region=cems_boundary,
    scale=30,          # Sentinel-2 native resolution is 10m/20m; 30m speeds things up
    numPixels=1500,    # 1,500 to 2,000 pixels is more than enough for clean plots!
    seed=42,
    geometries=False
)

# 2. Extract features safely into a Pandas DataFrame via standard GEE dictionary parsing
data_dict = sampled_fc.getInfo()
df = pd.DataFrame([feat['properties'] for feat in data_dict['features']]).dropna()

print(f"[INFO] Successfully sampled and loaded {len(df)} pixels inside the CEMS burn perimeter.")

# 3. Generate thesis plots
plot_dnbr_histogram(df)
plot_ndvi_scatterplot(df)

# Save once loaded so you can skip re-downloading
df.to_csv("resources/sampled_burn_data.csv", index=False)

# for local instant loads:
# import pandas as pd
# df = pd.read_csv("resources/sampled_burn_data.csv")